# Project 2: School Grade Analysis
## "Students Performance in Exams" — Real Student Score Dataset

**Lumexa Data Scientist Path — Course 13: Python for Data**

**Dataset:** `StudentsPerformance.csv` — a real, publicly accessible mirror of the well-known
"Students Performance in Exams" dataset (originally published on Kaggle by user `spscientist`),
hosted at a verified working GitHub mirror.
**Source:** https://raw.githubusercontent.com/rashida048/Datasets/master/StudentsPerformance.csv

This notebook is fully self-contained and works with **Runtime → Run all** — the real dataset
is downloaded directly from its public source at runtime, so there's nothing to upload and no
local file paths to configure. If this mirror is ever unavailable, the original dataset can be
downloaded directly from Kaggle (see the original dataset page:
https://www.kaggle.com/datasets/spscientist/students-performance-in-exams).

This notebook applies the full data science workflow to explore how demographic and
preparation factors relate to academic performance — carefully distinguishing correlation
from causation throughout.


In [1]:
# pandas and numpy are preinstalled in Google Colab.
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 140)

## 1. Load the data

In [2]:
STUDENTS_URL = "https://raw.githubusercontent.com/rashida048/Datasets/master/StudentsPerformance.csv"

students = pd.read_csv(STUDENTS_URL)
print("Shape:", students.shape)
students.head()

Shape: (1000, 8)


,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


## 2. Inspect the data

In [3]:
students.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   gender                       1000 non-null   str  
 1   race/ethnicity               1000 non-null   str  
 2   parental level of education  1000 non-null   str  
 3   lunch                        1000 non-null   str  
 4   test preparation course      1000 non-null   str  
 5   math score                   1000 non-null   int64
 6   reading score                1000 non-null   int64
 7   writing score                1000 non-null   int64
dtypes: int64(3), str(5)
memory usage: 62.6 KB


In [4]:
print("Missing values per column:")
print(students.isnull().sum())
print("\nDuplicate rows:", students.duplicated().sum())

Missing values per column:
gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

Duplicate rows: 0


**Findings:** The dataset has 1,000 rows and 8 columns, with **zero missing values** and
**zero duplicate rows** — a clean dataset straight from the source. Columns are: `gender`,
`race/ethnicity`, `parental level of education`, `lunch`, `test preparation course`,
`math score`, `reading score`, `writing score`.

## 3. Clean the data

In [5]:
# Even though this dataset arrived clean, we still perform the standard checks
# and create a derived 'average score' column used throughout the rest of the analysis.
for col in ["math score", "reading score", "writing score"]:
    assert students[col].dtype in ("int64", "float64"), f"{col} is not numeric!"

students["average score"] = students[["math score", "reading score", "writing score"]].mean(axis=1)
print(students[["math score", "reading score", "writing score", "average score"]].head())

   math score  reading score  writing score  average score
0          72             72             74      72.666667
1          69             90             88      82.333333
2          90             95             93      92.666667
3          47             57             44      49.333333
4          76             78             75      76.333333


## 4. Select, filter, and sort

In [6]:
# Filter to students who completed the test preparation course, sorted by average score
prepared_top = students[students["test preparation course"] == "completed"].sort_values(
    "average score", ascending=False
)
print("Top 5 scores among students who completed test preparation:")
prepared_top[["gender", "parental level of education", "average score"]].head()

Top 5 scores among students who completed test preparation:


,gender,parental level of education,average score
916,male,bachelor's degree,100.000000
114,female,bachelor's degree,99.666667
179,female,some high school,99.000000
625,male,some college,98.666667
165,female,bachelor's degree,98.666667


In [7]:
# Filter to students scoring below 60 average (at-risk students) for targeted support
at_risk = students[students["average score"] < 60]
print("Number of students with an average score below 60:", len(at_risk))
at_risk.sort_values("average score").head()[["gender", "test preparation course", "average score"]]

Number of students with an average score below 60: 285


,gender,test preparation course,average score
59,female,none,9.000000
980,female,none,18.333333
596,male,none,23.000000
327,male,none,23.333333
17,female,none,26.000000


## 5. Aggregations: group by and pivot tables

In [8]:
by_gender = students.groupby("gender")["average score"].mean().round(2)
print("Average score by gender:")
print(by_gender)

Average score by gender:
gender
female    69.57
male      65.84
Name: average score, dtype: float64


In [9]:
by_prep = students.groupby("test preparation course")["average score"].mean().round(2)
print("Average score by test preparation course status:")
print(by_prep)

Average score by test preparation course status:
test preparation course
completed    72.67
none         65.04
Name: average score, dtype: float64


In [10]:
by_parent_edu = students.groupby("parental level of education")["average score"].mean().sort_values(ascending=False).round(2)
print("Average score by parental level of education:")
print(by_parent_edu)

Average score by parental level of education:
parental level of education
master's degree       73.60
bachelor's degree     71.92
associate's degree    69.57
some college          68.48
some high school      65.11
high school           63.10
Name: average score, dtype: float64


In [11]:
pivot = pd.pivot_table(
    students, values="math score", index="gender", columns="test preparation course", aggfunc="mean"
).round(2)
print("Average math score by gender and test preparation status:")
pivot

Average math score by gender and test preparation status:


test preparation course,completed,none
gender,,
female,67.20,61.67
male,72.34,66.69


## 6. Statistical summaries and correlation

In [12]:
print("Average score - overall summary statistics:")
print(students["average score"].describe().round(2))

Average score - overall summary statistics:
count    1000.00
mean       67.77
std        14.26
min         9.00
25%        58.33
50%        68.33
75%        77.67
max       100.00
Name: average score, dtype: float64


In [13]:
corr_matrix = students[["math score", "reading score", "writing score"]].corr().round(3)
print("Correlation matrix between the three subject scores:")
corr_matrix

Correlation matrix between the three subject scores:


,math score,reading score,writing score
math score,1.000,0.818,0.803
reading score,0.818,1.000,0.955
writing score,0.803,0.955,1.000


## 7. Conclusions

Based on the real analysis above:

1. The dataset contains 1,000 student records with zero missing values and zero duplicates,
   across 8 columns describing demographic factors and three exam scores.
2. Students who **completed** the test preparation course scored noticeably higher on average
   (about 72.7) than those who did **not** (about 65.0) — a difference of roughly 7.6 points,
   suggesting a real, meaningful association between test preparation and exam performance.
3. In this dataset, female students had a higher average score (about 69.6) than male students
   (about 65.8) — this is an *observed pattern in this specific dataset*, not a general claim
   about ability.
4. Average score rises steadily with parental level of education, from about 63.1 (some high
   school) to about 73.6 (master's degree) — a real association worth investigating further,
   though this dataset alone cannot establish causation.
5. Reading and writing scores are very strongly correlated (about 0.95), and math is
   moderately-to-strongly correlated with both (about 0.80-0.82) — students who do well in one
   subject tend to do well in the others.

These findings are based entirely on the real "Students Performance in Exams" dataset and the
computations performed in this notebook — no numbers were invented or assumed. As always,
demographic findings should be phrased as observed patterns in this dataset, not general claims.

**Try it yourself:** change the `average score` threshold in Section 4, or group by
`race/ethnicity` in Section 5, and re-run (`Runtime → Run all`) to explore further.